In [7]:
# llm_interface.py
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
GNN_API_URL = "http://localhost:8000/match"

def ask_ollama(prompt, model="llama3.2:3b"):
    response = requests.post(
        OLLAMA_URL,
        json={"model": model, "prompt": prompt, "stream": False},
    )
    data = response.json()
    return data.get("response", "").strip()

import re

def get_entities(user_input):
    prompt = f"""
    You are an expert in pharmacology. From the text below, extract:
    1. The list of drugs or medicine names mentioned
    2. The list of side effects or symptoms mentioned

    Return your answer strictly in JSON format like:
    {{
      "drugs": ["DrugA", "DrugB"],
      "side_effects": ["nausea", "dizziness"]
    }}

    Text: "{user_input}"
    """

    try:
        output = ask_ollama(prompt)
        # 🔍 Find JSON inside the model output using regex
        json_match = re.search(r"\{.*\}", output, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            data = json.loads(json_str)
            return data
        else:
            raise ValueError("No JSON found in response.")
    except Exception as e:
        print("Parsing error. Raw output:", output)
        print("Error:", e)
        return {"drugs": [], "side_effects": []}


def main():
    print("🧠 Drug Side-Effect Assistant (HODDI + Ollama)")
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ["exit", "quit"]:
            break

        entities = get_entities(user_input)
        print("🔍 Extracted:", entities)

        side_effects = entities["side_effects"]
        if not side_effects:
            print("❌ No recognizable side effects.")
            continue

        result = requests.get(GNN_API_URL, params={"symptoms": side_effects})
        matches = result.json().get("matches", [])

        reasoning_prompt = f"""
        The user reported side effects: {side_effects}.
        The GNN model found these possible drug matches: {matches}.
        Explain which one could be the cause and why.
        """
        explanation = ask_ollama(reasoning_prompt)
        print("\n💊 Explanation:", explanation)

if __name__ == "__main__":
    main()


🧠 Drug Side-Effect Assistant (HODDI + Ollama)
🔍 Extracted: {'drugs': [], 'side_effects': ['dizziness', 'nausea']}

💊 Explanation: Based on the information provided, it appears that there may be a mismatch between the reported side effects of the user and the possible drug matches found by the GNN (Graph Neural Network) model.

The user reported experiencing dizziness and nausea as potential side effects, which suggests that these symptoms are related to the consumption of a particular drug or medication. However, the GNN model did not find any possible drug matches for these side effects, indicating that the model was unable to identify a correlation between the user's symptoms and a specific medication.

Given this discrepancy, it is possible that the reported side effects are not directly related to a commonly prescribed medication. Here are a few potential explanations:

1. **Misattribution of side effects**: The user may be misattributing their symptoms to the drug they consumed, w

KeyboardInterrupt: Interrupted by user

In [10]:
import torch
side_effect_embeddings = torch.load(r"C:\Users\Manav\OneDrive\Desktop\MIT\3rd Year\Sem V\ANN\Project\HODDI\Data\side_effect_embeddings.pt")
print(list(side_effect_embeddings.keys())[:10])


['names', 'embeddings']


C:\Users\Manav\AppData\Local\Temp\ipykernel_16264\129019351.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  side_effect_embeddings = torch.load(r"C:\Users\Manav\OneDrive